# Sampling Strategy: Generation and Balancing

This notebook implements a new sampling strategy:
1. Generate synthetic Class 0 (non-fire) points to create an imbalanced dataset (70% Class 1, 30% Class 0).
2. Filter out generated points that fall on sea (using elevation data).
3. Oversample Class 0 to create a fully balanced dataset (50/50).

In [1]:
import pandas as pd
import numpy as np
import rasterio
from sklearn.utils import resample
import os

# Set random seed for reproducibility
np.random.seed(42)

In [2]:
# Load the dataset
input_path = '../results/fires_merged_all_features.csv'
df = pd.read_csv(input_path)

print("Original Dataset Shape:", df.shape)
print("Class Distribution:\n", df['class'].value_counts())

# Ensure only class 1 exists (as per previous steps)
df = df[df['class'] == 1].copy()
print("Filtered Dataset Shape (Class 1 only):", df.shape)

Original Dataset Shape: (3824, 42)
Class Distribution:
 class
1    3824
Name: count, dtype: int64
Filtered Dataset Shape (Class 1 only): (3824, 42)


In [3]:
# Calculate number of class 0 samples needed
# Target: 70% Class 1, 30% Class 0
# N0 = N1 * (30/70)
n_class_1 = len(df)
n_class_0_target = int(n_class_1 * (30/70))
print(f"Target Class 0 samples: {n_class_0_target}")

# We generate more initially to account for sea points removal
n_generate = n_class_0_target * 3

# Get bounding box
min_lon, max_lon = df['longitude'].min(), df['longitude'].max()
min_lat, max_lat = df['latitude'].min(), df['latitude'].max()

print(f"Bounding Box: Lon ({min_lon}, {max_lon}), Lat ({min_lat}, {max_lat})")

# Generate random coordinates
synthetic_data = {
    'longitude': np.random.uniform(min_lon, max_lon, n_generate),
    'latitude': np.random.uniform(min_lat, max_lat, n_generate),
    'class': 0
}

# Generate random values for other columns based on min-max of Class 1
# Note: This generates random feature combinations (noise) for Class 0.
for col in df.columns:
    if col not in ['longitude', 'latitude', 'class']:
        if pd.api.types.is_numeric_dtype(df[col]):
            min_val = df[col].min()
            max_val = df[col].max()
            synthetic_data[col] = np.random.uniform(min_val, max_val, n_generate)
        else:
            # For categorical, sample randomly
            synthetic_data[col] = np.random.choice(df[col].unique(), n_generate)

df_synthetic = pd.DataFrame(synthetic_data)
print("Generated Synthetic Samples (Candidates):", df_synthetic.shape)

Target Class 0 samples: 1638
Bounding Box: Lon (-2.18114, 11.11035), Lat (33.0164, 37.32346)
Generated Synthetic Samples (Candidates): (4914, 42)


In [7]:
# Path to elevation raster
elevation_raster_path = '../dataset/elevation_dataset/be15_grd'

# Function to check if point is on land (valid elevation)
def filter_land_points(df, raster_path):
    land_indices = []
    try:
        with rasterio.open(raster_path) as src:
            print(f"Raster NoData Value: {src.nodata}")
            
            # Check a known sea point from sea.csv
            # -1.4228, 36.06831
            sea_point = [(-1.4228, 36.06831)]
            sea_elev = list(src.sample(sea_point))[0][0]
            print(f"Elevation of known sea point (-1.4228, 36.06831): {sea_elev}")
            
            # Sample elevation at coordinates
            coords = [(x, y) for x, y in zip(df['longitude'], df['latitude'])]
            elevations = list(src.sample(coords))
            
            for i, val in enumerate(elevations):
                # Check if value is not NoData AND not 0 (assuming 0 is sea)
                if src.nodata is not None:
                    if val[0] != src.nodata and val[0] != 0:
                        land_indices.append(i)
                else:
                    if val[0] > -10000 and val[0] != 0:
                        land_indices.append(i)
    except Exception as e:
        print(f"Error reading raster: {e}")
        return df # Return all if raster fails (fallback)
                
    return df.iloc[land_indices].copy()

print("Filtering sea points using elevation raster...")
df_synthetic_land = filter_land_points(df_synthetic, elevation_raster_path)
print("Synthetic Samples after Sea Removal:", df_synthetic_land.shape)

# If we have too many, sample down to target
if len(df_synthetic_land) > n_class_0_target:
    df_synthetic_final = df_synthetic_land.sample(n=n_class_0_target, random_state=42)
else:
    print(f"Warning: Not enough land points generated ({len(df_synthetic_land)} < {n_class_0_target}). Using all available.")
    df_synthetic_final = df_synthetic_land

print("Final Class 0 Samples:", df_synthetic_final.shape)

Filtering sea points using elevation raster...
Raster NoData Value: -32768.0
Elevation of known sea point (-1.4228, 36.06831): 0
Synthetic Samples after Sea Removal: (3970, 42)
Final Class 0 Samples: (1638, 42)


In [8]:
# Concatenate to create Imbalanced Dataset
imbalanced_df = pd.concat([df, df_synthetic_final], ignore_index=True)
imbalanced_df = imbalanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Imbalanced Dataset Shape:", imbalanced_df.shape)
print("Class Distribution:\n", imbalanced_df['class'].value_counts(normalize=True))

# Save
imbalanced_df.to_csv('../results/imbalanced_dataset.csv', index=False)
print("Saved ../results/imbalanced_dataset.csv")

Imbalanced Dataset Shape: (5462, 42)
Class Distribution:
 class
1    0.70011
0    0.29989
Name: proportion, dtype: float64
Saved ../results/imbalanced_dataset.csv


In [ ]:
# Separate majority and minority classes
df_majority = imbalanced_df[imbalanced_df['class'] == 1]
df_minority = imbalanced_df[imbalanced_df['class'] == 0]

# Upsample minority class
df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=len(df_majority),    # to match majority class
                                 random_state=42) # reproducible results

# Combine majority class with upsampled minority class
balanced_df = pd.concat([df_majority, df_minority_upsampled])

# Shuffle
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced Dataset Shape:", balanced_df.shape)
print("Class Distribution:\n", balanced_df['class'].value_counts())

# Save
balanced_df.to_csv('../results/balanced_dataset.csv', index=False)
print("Saved ../results/balanced_dataset.csv")

Balanced Dataset Shape: (7648, 42)
Class Distribution:
 class
1    3824
0    3824
Name: count, dtype: int64
Saved ../results/balanced_dataset.csv


: 